In [ ]:
import itertools as it

import boto3
import botocore
from cliffs_delta import cliffs_delta
import numpy as np
import pandas as pd
from pandas.util import hash_pandas_object
from scipy import stats as scipy_stats
import seaborn as sns
import seaborn.objects as so
from teeplot import teeplot as tp


In [ ]:
from dishpylib.pyhelpers import make_outattr_metadata
from dishpylib.pyhelpers import print_runtime


In [ ]:
print_runtime()


In [ ]:
teeplot_subdir = "2026-08-18-complexty-fitness-correlation"


# get data


In [ ]:
s3_handle = boto3.resource(
    's3',
    region_name="us-east-2",
    config=botocore.config.Config(
        signature_version=botocore.UNSIGNED,
    ),
)
bucket_handle = s3_handle.Bucket('prq49')

series_profiles, = bucket_handle.objects.filter(
    Prefix=f'endeavor=16/series-profiles/stage=8+what=elaborated/',
)


In [ ]:
df = pd.read_csv(
    f's3://prq49/{series_profiles.key}',
    compression='xz',
)
dfdigest = '{:x}'.format( hash_pandas_object( df ).sum() )
dfdigest


In [ ]:
def make_outattr_metadata():
    return {}


In [ ]:
for stint in df['Stint'].unique():
    exec(f'df{stint} = df[ df["Stint"] == {stint} ]')


In [ ]:
dfm10 = df[ df['Stint'] % 10 == 9 ]  # rerun for replication/complete data @ stint 9, 19, 29, 39, 49, 59, 69, 79, 89, 99


In [ ]:
s3_handle = boto3.resource(
    "s3",
    region_name="us-east-2",
    config=botocore.config.Config(
        signature_version=botocore.UNSIGNED,
    ),
)
bucket_handle = s3_handle.Bucket("prq49")

dfnone = pd.concat(
    [
        pd.read_csv(f"s3://prq49/{item.key}")
        for item in bucket_handle.objects.filter(
            Prefix=f"endeavor=16/external-competitions/stage=2+what=collated/",
        )
    ],
    ignore_index=True,
)
dfnone["kind"] = "none"
print(dfnone["Competition Stint"].unique())
print(dfnone["Competition Series"].unique())


In [ ]:
dfx = dfnone[
    dfnone['Competition Stint'] % 10 == 9
].copy()
dfx["Fitness Differential Focal Sign"] = np.sign(
    dfx["Fitness Differential Focal"],
)
dfx["Series"] = dfx["Competition Series"]
dfx["Stint"] = dfx["Competition Stint"]
dfx["flip series"] = dfx["genome series"] * (dfx["Root ID"] == 1)
dfx["Competitor"] = (
    dfx
    .groupby(["Stint", "Series", "kind", "Competition Repro"])
    ["flip series"]
    .transform("max")
)
dfx = dfx[
    dfx["Root ID"] == 0
].groupby(["Stint", "Series", "kind", "Competitor"])[
    "Focal Prevalence"
].mean().round().reset_index(drop=False)
dfx


In [ ]:
dfj = dfx.join(
    dfm10[
        [
            "Stint",
            "Series",
            "Fitness Complexity",
            "Flagged Advantageous Sites",
            "Flagged Deleterious Sites",
            "Cardinal Interface Complexity",
            "Cell Interface Complexity",
        ]
    ].set_index(
        ["Stint", "Series"]
    ),
    on=["Stint", "Series"],
    how="inner",
).reset_index(drop=False)
dfj["Favored"] = dfj["Focal Prevalence"] > 0.5
dfj


In [ ]:
palette = [
    sns.color_palette("Dark2")[2],
    sns.color_palette("Dark2")[4],
    sns.color_palette("Dark2")[0],
]


In [ ]:
light_palette=[
    "#a3cf73",
    "#6dceb1",
]
dark_palette=[
    "#4a772f",
    "#137177",
]


## Fitness


In [ ]:
for (y, when, agg) in it.product(
    ["Focal Prevalence"],
    ["early", "late", "all"],
    [True, False],
):
    # 1. Base Aggregation
    dfjx = dfj.groupby(
        ["Series", "kind", "Stint"],
    ).mean().reset_index(drop=False)

    # 2. Separate Baseline (eco) and Target (self)
    df_none = dfjx[dfjx["kind"] == "none"].copy()

    # 5. Prepare Plot Data (Aggregated)
    whenisin = {
        "early": range(9, 51, 10),
        "late": range(59, 101, 10),
        "all": range(9, 101, 10),
    }[when]

    df_plot = df_none[
        df_none["Stint"].isin(whenisin)
    ].reset_index(drop=True)
    df_plot["Phenotype Complexity"] = df_plot["Cardinal Interface Complexity"]

    print(df_plot["Phenotype Complexity"].isnull().sum())  # Check for NaN values in Phenotype Complexity

    if agg:
        df_plot = df_plot.groupby(["Series"]).mean().reset_index(drop=False)

    # 6. Calculate Regression Stats
    slope, intercept, r_value, p_value, std_err = scipy_stats.linregress(
        df_plot["Phenotype Complexity"],
        df_plot[y]
    )

    # Determine significance
    is_significant = p_value < 0.05

    # Format the annotation text with bold if significant
    if is_significant:
        stats_text = (
            f"$\\mathbf{{R^2 = {r_value**2:.2f}}}$\n"
            f"$\\mathbf{{p = {p_value:{'.1e' if p_value < 0.01 else '.3f'}}}}$"
        )
    else:
        stats_text = (
            f"$R^2 = {r_value**2:.2f}$\n"
            f"$p = {p_value:{'.1e' if p_value < 0.01 else '.3f'}}$"
        )

    # 7. Plot
    with tp.teed(
        sns.lmplot,
        data=df_plot,
        x="Phenotype Complexity",
        y=y,
        line_kws={"lw": 1.0, "alpha": 0.7, "color": "darkkhaki", "linestyle": '-' if is_significant else ':'},
        scatter_kws={"color": "olive", "alpha": 0.3, "clip_on": False, "s": 5},
        facet_kws=dict(
            sharex=False,
            sharey=True,
        ),
        teeplot_outattrs={"agg": agg, "when": when},
        teeplot_subdir=teeplot_subdir,
    ) as g:
        g.figure.set_size_inches(2, 1.2)  # Adjusted size for single plot
        g.set_xlabels("Phenotype Complexity")
        g.set_ylabels("Fitness")

        # Add a horizontal reference line
        ax = g.axes.flat[0]
        ax.set_ylim(-0.02, 1.02)

        # Add the stats annotation to the plot
        # Anchored to upper left or right depending on data layout
        g.figure.subplots_adjust(left=0.2)
        ax.text(
            1, 0,
            stats_text,
            transform=ax.transAxes,
            horizontalalignment='right',
            verticalalignment='bottom',
            fontsize=8,
            color="black" if is_significant else "lightgray",
            weight="bold" if is_significant else "normal",
        )


In [ ]:
for (y, when, agg) in it.product(
    ["Focal Prevalence"],
    ["early", "late", "all"],
    [True, False],
):
    # 1. Base Aggregation
    dfjx = dfj.groupby(
        ["Series", "kind", "Stint"],
    ).mean().reset_index(drop=False)

    # 2. Separate Baseline (eco) and Target (self)
    df_none = dfjx[dfjx["kind"] == "none"].copy()

    # 5. Prepare Plot Data (Aggregated)
    whenisin = {
        "early": range(9, 51, 10),
        "late": range(59, 101, 10),
        "all": range(9, 101, 10),
    }[when]
    df_none["Genetic Complexity"] = df_none["Flagged Advantageous Sites"]

    print(df_none[
        df_none["Genetic Complexity"].isnull()
    ])  # Check for NaN values in Genetic Complexity
    # Series 1029/stint 9 has NaN values because all sites were detected as phenotype-equivalent
    # Thus, it can be filled as zero
    df_none["Genetic Complexity"].fillna(0, inplace=True)
    print(df_none["Genetic Complexity"].isnull().sum())  # Check again for NaN values in Genetic Complexity

    df_plot = df_none[
        df_none["Stint"].isin(whenisin)
    ].reset_index(drop=True)
    if agg:
        df_plot = df_plot.groupby(["Series"]).mean().reset_index(drop=False)

    # 6. Calculate Regression Stats
    slope, intercept, r_value, p_value, std_err = scipy_stats.linregress(
        df_plot["Genetic Complexity"],
        df_plot[y]
    )

    # Determine significance
    is_significant = p_value < 0.05

    # Format the annotation text with bold if significant
    if is_significant:
        stats_text = (
            f"$\\mathbf{{R^2 = {r_value**2:.2f}}}$\n"
            f"$\\mathbf{{p = {p_value:{'.1e' if p_value < 0.01 else '.3f'}}}}$"
        )
    else:
        stats_text = (
            f"$R^2 = {r_value**2:.2f}$\n"
            f"$p = {p_value:{'.1e' if p_value < 0.01 else '.3f'}}$"
        )

    # 7. Plot
    with tp.teed(
        sns.lmplot,
        data=df_plot,
        x="Genetic Complexity",
        y=y,
        line_kws={"lw": 1.0, "alpha": 0.7, "color": "darkkhaki", "linestyle": '-' if is_significant else ':'},
        scatter_kws={"color": "olive", "alpha": 0.3, "clip_on": False, "s": 5},
        facet_kws=dict(
            sharex=False,
            sharey=True,
        ),
        teeplot_outattrs={"agg": agg, "when": when},
        teeplot_subdir=teeplot_subdir,
    ) as g:
        g.figure.set_size_inches(2, 1.2)  # Adjusted size for single plot
        g.set_xlabels("Genetic Complexity")
        g.set_ylabels("Fitness")

        # Add a horizontal reference line
        ax = g.axes.flat[0]
        ax.set_ylim(-0.02, 1.02)

        # Add the stats annotation to the plot
        # Anchored to upper left or right depending on data layout
        g.figure.subplots_adjust(left=0.2)
        ax.text(
            1, 0,
            stats_text,
            transform=ax.transAxes,
            horizontalalignment='right',
            verticalalignment='bottom',
            fontsize=8,
            color="black" if is_significant else "lightgray",
            weight="bold" if is_significant else "normal",
        )
